# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [17]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import optuna
import optuna.visualization as vis
import time

import scipy.stats as st

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error, make_scorer

from sklearn.model_selection import cross_val_score

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix, f1_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [5]:
#drop rows containing any missing value (be careful, create a new df otherwise this does not work)
spaceship_clean = spaceship.dropna(axis="rows")
#replace cabin values
spaceship_clean['Cabin']=spaceship_clean['Cabin'].str[0]
#Drop PassengerId and Name
spaceship_clean = spaceship_clean.drop(columns = ["PassengerId","Name"])
# all columns
features = spaceship_clean.drop(columns=['Transported'], axis=1)
target = spaceship_clean["Transported"]

#here we make sure  that we gather all unique categorical values from our df columns in 'categories'. This way we avoid any missing and/or additional value
ohe = OneHotEncoder(sparse_output=False, drop='first', categories = [spaceship_clean[col].unique().tolist() for col in spaceship_clean.select_dtypes(include="object").columns])

#Perform Train Test Split
X_train, X_test, y_train, y_test = train_test_split(features,target, test_size = 0.20, random_state=0)

# X train dummies for categorical columns
X_train_cat = X_train.select_dtypes(include="object") 
X_test_cat = X_test.select_dtypes(include="object") 
ohe.fit(X_train_cat)  
X_train_cat_trans_np = ohe.transform(X_train_cat)
X_train_cat_trans_df = pd.DataFrame(X_train_cat_trans_np, columns=ohe.get_feature_names_out(), index=X_train_cat.index)

X_test_cat_trans_np = ohe.transform(X_test_cat)
X_test_cat_trans_df = pd.DataFrame(X_test_cat_trans_np, columns=ohe.get_feature_names_out(), index=X_test_cat.index)

# X train for numerical columns
X_train_num = X_train.select_dtypes(include="number") 
X_test_num = X_test.select_dtypes(include="number") 

# concatenation of X_cat & X_num before scaling
X_train_transformed = pd.concat([X_train_num.reset_index(drop=True), pd.DataFrame(X_train_cat_trans_df).reset_index(drop=True)], axis = 1)
X_test_transformed = pd.concat([X_test_num.reset_index(drop=True), pd.DataFrame(X_test_cat_trans_df).reset_index(drop=True)], axis = 1)

# apply scaling for all
# creation of the instance
std_scaler = StandardScaler()

# application to the values
std_scaler.fit(X_train_transformed)

# transformation of the data
X_train_transformed_np = std_scaler.transform(X_train_transformed)
X_test_transformed_np  = std_scaler.transform(X_test_transformed)

# conversion into df
X_train_transformed_np_df = pd.DataFrame(X_train_transformed_np, columns=X_train_transformed.columns, index=X_train_transformed.index)
X_test_transformed_np_df  = pd.DataFrame(X_test_transformed_np, columns=X_test_transformed.columns, index=X_test_transformed.index)

X_test_transformed_np_df.shape

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_10512\3386669263.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_clean['Cabin']=spaceship_clean['Cabin'].str[0]


(1322, 19)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [10]:
knn_final = KNeighborsClassifier(n_neighbors=10)

knn_final.fit(X_train_transformed_np_df, y_train)

with open("knn_final.pkl", "wb") as file:
    pickle.dump(knn_final, file)

print(f"on train, model accuracy is: {knn_final.score(X_train_transformed_np_df, y_train)*100:.2f}%")
print(f"on test, model accuracy is: {knn_final.score(X_test_transformed_np_df, y_test)*100:.2f}%")

y_pred_knn = knn_final.predict(X_test_transformed) #to get the result values of y! then you can concate on y_test, and X_test
print(f"MAE {mean_absolute_error(y_pred_knn, y_test): .2f}")
print(f"MSE {mean_squared_error(y_pred_knn, y_test): .2f}")
print(f"RMSE {root_mean_squared_error(y_pred_knn, y_test): .2f}")
print(f"R2 score {knn_final.score(X_test_transformed, y_test): .2f}")

on train, model accuracy is: 81.25%
on test, model accuracy is: 76.93%
MAE  0.37
MSE  0.37
RMSE  0.61
R2 score  0.63


- Evaluate your model

In [11]:
# the KNN model was the best regarding the results of the R2 Score of 63%. 

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [8]:
# https://www.datasklr.com/select-classification-methods/k-nearest-neighbors

# KNN parameters
parameter_grid = {
    'n_neighbors': (3, 5, 7, 9, 11, 15), # number of neighbors searched
    'leaf_size': (30,50), # number of points within a leaf
    'p': (1,2), #distance formulas 
    'weights': ('uniform', 'distance')} # neighbor's weight

# We create an instance or our machine learning model
kc = KNeighborsClassifier(algorithm='kd_tree') 
# KD-Tree due to the size of the train dataset around 5000 points

# We need to set this two variables to be able to compute a confidence interval
confidence_level = 0.95 # confidence_level = 1 - alpha
folds = 10



- Run Grid Search

In [9]:
# Now we need to create an intance of the GridSearchCV class
# The option "scoring", alongside with new_metric ="make_scorer" allows you to evaluate the model performance using a different error metrics.
gs = GridSearchCV(kc, param_grid=parameter_grid, cv=folds, verbose=10) # Here the "cv" allows you to define the number of folds to use.

start_time = time.time()
gs.fit(X_train_transformed_np_df, y_train)
end_time = time.time()

print("\n")
print(f"Time taken to find the best combination of hyperparameters among the given ones: {end_time - start_time: .4f} seconds")
print("\n")




Fitting 10 folds for each of 48 candidates, totalling 480 fits
[CV 1/10; 1/48] START leaf_size=30, n_neighbors=3, p=1, weights=uniform.........
[CV 1/10; 1/48] END leaf_size=30, n_neighbors=3, p=1, weights=uniform;, score=0.733 total time=   0.0s
[CV 2/10; 1/48] START leaf_size=30, n_neighbors=3, p=1, weights=uniform.........
[CV 2/10; 1/48] END leaf_size=30, n_neighbors=3, p=1, weights=uniform;, score=0.788 total time=   0.0s
[CV 3/10; 1/48] START leaf_size=30, n_neighbors=3, p=1, weights=uniform.........
[CV 3/10; 1/48] END leaf_size=30, n_neighbors=3, p=1, weights=uniform;, score=0.760 total time=   0.0s
[CV 4/10; 1/48] START leaf_size=30, n_neighbors=3, p=1, weights=uniform.........
[CV 4/10; 1/48] END leaf_size=30, n_neighbors=3, p=1, weights=uniform;, score=0.773 total time=   0.0s
[CV 5/10; 1/48] START leaf_size=30, n_neighbors=3, p=1, weights=uniform.........
[CV 5/10; 1/48] END leaf_size=30, n_neighbors=3, p=1, weights=uniform;, score=0.754 total time=   0.0s
[CV 6/10; 1/48] S

- Evaluate your model

In [10]:
print(f"The best combination of hyperparameters has been: {gs.best_params_}")
print(f"The R2 is: {gs.best_score_: .4f}")

The best combination of hyperparameters has been: {'leaf_size': 30, 'n_neighbors': 15, 'p': 1, 'weights': 'uniform'}
The R2 is:  0.7810


In [11]:
# Wow! this is so cool! therefore my best KNN model has the following metrics according to grid search: a box (leaf size) with 30 points better than more, but a lot of neigbors would be better to consider, manhattant distance and uniform wieght for any neighbor!

In [13]:
# so the next step is to include these params to my model to see the accuracy. 
# I want also to determine additional metrics

In [18]:
knn_optimizor = KNeighborsClassifier(n_neighbors=15, weights = 'uniform', 
                                 p = 1, algorithm = 'kd_tree', leaf_size = 30) 

knn_optimizor.fit(X_train_transformed_np_df, y_train)

with open("knn_optimizor.pkl", "wb") as file:
    pickle.dump(knn_optimizor, file)

print(f"on train, model accuracy is: {knn_optimizor.score(X_train_transformed_np_df, y_train)*100:.2f}%")
print(f"on test, model accuracy is: {knn_optimizor.score(X_test_transformed_np_df, y_test)*100:.2f}%")

y_pred_knn_opt = knn_optimizor.predict(X_test_transformed_np_df) #to get the result values of y! then you can concate on y_test, and X_test
print(f"MAE {mean_absolute_error(y_pred_knn_opt, y_test): .2f}")
print(f"MSE {mean_squared_error(y_pred_knn_opt, y_test): .2f}")
print(f"RMSE {root_mean_squared_error(y_pred_knn_opt, y_test): .2f}")
print(f"R2 score {knn_optimizor.score(X_test_transformed_np_df, y_test): .2f}")

y_pred_test_log = knn_optimizor.predict(X_test_transformed_np_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

on train, model accuracy is: 80.92%
on test, model accuracy is: 78.37%
MAE  0.22
MSE  0.22
RMSE  0.47
R2 score  0.78
              precision    recall  f1-score   support

       False       0.79      0.78      0.78       661
        True       0.78      0.79      0.78       661

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322

